In [4]:
import mysql.connector as mysql

connection = mysql.connect(
    host="localhost",
    user="root",
    password="1234",
    database="fraud_detection_gnn"
)

cursor = connection.cursor()
print("Connected successfully!")

Connected successfully!


In [5]:
query = """
SELECT 'CUSTOMER' AS Table_Name, COUNT(*) AS Records FROM CUSTOMER
UNION ALL
SELECT 'BANK', COUNT(*) FROM BANK
UNION ALL
SELECT 'ACCOUNT', COUNT(*) FROM ACCOUNT
UNION ALL
SELECT 'DEVICE', COUNT(*) FROM DEVICE
UNION ALL
SELECT 'UPI_ACCOUNT', COUNT(*) FROM UPI_ACCOUNT
UNION ALL
SELECT 'MERCHANT', COUNT(*) FROM MERCHANT
UNION ALL
SELECT 'TRANSACTION', COUNT(*) FROM TRANSACTION
UNION ALL
SELECT 'FRAUD_CASE', COUNT(*) FROM FRAUD_CASE;"""

print("Executing query to get record counts from all tables...")
cursor.execute(query)
cursor_results = cursor.fetchall()
print("Query results:")
for row in cursor_results:
    print(f"Table: {row[0]}, Records: {row[1]}")

Executing query to get record counts from all tables...
Query results:
Table: CUSTOMER, Records: 250
Table: BANK, Records: 10
Table: ACCOUNT, Records: 350
Table: DEVICE, Records: 300
Table: UPI_ACCOUNT, Records: 250
Table: MERCHANT, Records: 100
Table: TRANSACTION, Records: 9000
Table: FRAUD_CASE, Records: 2000


In [10]:
import pandas as pd
query = """

SELECT
    t.Transaction_ID,
    t.Amount,
    t.Channel,
    t.Transaction_Timestamp,
    HOUR(t.Transaction_Timestamp) AS Transaction_Hour,
    DAYOFWEEK(t.Transaction_Timestamp) AS Transaction_Day,
    t.Is_International,
    t.Is_Reversal,
    t.Failure_Count,

    sa.Account_ID AS Sender_Account,
    sa.Account_Type AS Sender_Account_Type,
    sa.Balance AS Sender_Balance,
    sa.Risk_Score AS Sender_Risk_Score,
    sa.KYC_Status AS Sender_KYC,

    ra.Account_ID AS Receiver_Account,
    ra.Account_Type AS Receiver_Account_Type,
    ra.Balance AS Receiver_Balance,
    ra.Risk_Score AS Receiver_Risk_Score,
    ra.KYC_Status AS Receiver_KYC,

    sc.City AS Sender_City,
    sc.State AS Sender_State,
    sc.Occupation AS Sender_Occupation,
    sc.KYC_Status AS Sender_Customer_KYC,
    sc.Identity_Verified AS Sender_Identity_Verified,
    sc.Gender AS Sender_Gender,
    TIMESTAMPDIFF(
        YEAR,
        sc.DOB,
        t.Transaction_Timestamp
    ) AS Sender_Age,

    rc.City AS Receiver_City,
    rc.State AS Receiver_State,
    rc.Occupation AS Receiver_Occupation,
    rc.KYC_Status AS Receiver_Customer_KYC,
    rc.Identity_Verified AS Receiver_Identity_Verified,
    rc.Gender AS Receiver_Gender,
    TIMESTAMPDIFF(
        YEAR,
        rc.DOB,
        t.Transaction_Timestamp
    ) AS Receiver_Age,

    d.Device_Type,
    d.OS,
    d.OS_Version,
    d.Login_Count,
    d.Failed_Login_Count,
    d.New_Device_Flag,
    d.Risk_Score AS Device_Risk_Score,

    m.Merchant_Category,
    m.Risk_Score AS Merchant_Risk_Score,

    CASE
        WHEN u.UPI_ID IS NOT NULL THEN 1
        ELSE 0
    END AS UPI_Used,

    u.PSP_Name,
    u.Registration_Date AS UPI_Registration_Date,

    f.Fraud_Type,
    COALESCE(f.Label, 0) AS Label

FROM TRANSACTION t

LEFT JOIN ACCOUNT sa
    ON t.Sender_Account_ID = sa.Account_ID

LEFT JOIN ACCOUNT ra
    ON t.Receiver_Account_ID = ra.Account_ID

LEFT JOIN CUSTOMER sc
    ON sa.User_ID = sc.User_ID

LEFT JOIN CUSTOMER rc
    ON ra.User_ID = rc.User_ID

LEFT JOIN DEVICE d
    ON t.Device_ID = d.Device_ID

LEFT JOIN MERCHANT m
    ON t.Merchant_ID = m.Merchant_ID

LEFT JOIN UPI_ACCOUNT u
    ON t.UPI_ID = u.UPI_ID

LEFT JOIN FRAUD_CASE f
    ON t.Transaction_ID = f.Transaction_ID;

"""

df = pd.read_sql(query, connection)

print(df.head())

C:\Users\Manthan\AppData\Local\Temp\ipykernel_24248\371898732.py:101: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


   Transaction_ID   Amount Channel Transaction_Timestamp  Transaction_Hour  \
0               1  1087.89    IMPS   2026-06-28 15:06:43                15   
1               2  1858.49    NEFT   2026-02-11 15:37:49                15   
2               3  3129.76    IMPS   2026-03-11 00:48:01                 0   
3               4   640.08     UPI   2026-09-15 16:42:58                16   
4               5  1380.45    IMPS   2026-01-23 16:19:31                16   

   Transaction_Day  Is_International  Is_Reversal  Failure_Count  \
0                1                 0            0              0   
1                4                 0            0              0   
2                4                 0            0              0   
3                3                 0            0              1   
4                6                 0            0              1   

   Sender_Account  ... Failed_Login_Count  New_Device_Flag  Device_Risk_Score  \
0             243  ...                 15